# Person 3: Clean FRED Economic Data

This notebook cleans the FRED series used by the economic and labor data workstream for the Ohio rent-versus-buy project.

Input raw files should be saved under `data/raw/fred/`:

- `MORTGAGE30US.csv` - 30-Year Fixed Rate Mortgage Average in the United States
- `CPIAUCSL.csv` - Consumer Price Index for All Urban Consumers: All Items in U.S. City Average
- `FEDFUNDS.csv` - Effective Federal Funds Rate

Output file:

- `data/processed/fred_economic_annual.csv`

Notes:

- Raw files are never edited.
- Weekly and monthly series are converted to annual means.
- Inflation is computed from annual average CPI as year-over-year percent change.
- 2025 may be partial depending on when the data was downloaded. Check the coverage table before treating 2025 as complete.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## 1. Define paths and series metadata

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "fred"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "fred_economic_annual.csv"

START_DATE = "2015-01-01"
END_DATE = "2025-12-31"

SERIES = {
    "MORTGAGE30US": {
        "filename": "MORTGAGE30US.csv",
        "clean_name": "mortgage_rate_pct",
        "description": "30-year fixed mortgage rate; annual mean of weekly observations",
    },
    "CPIAUCSL": {
        "filename": "CPIAUCSL.csv",
        "clean_name": "cpi",
        "description": "Consumer Price Index; annual mean of monthly observations",
    },
    "FEDFUNDS": {
        "filename": "FEDFUNDS.csv",
        "clean_name": "interest_rate_pct",
        "description": "Effective federal funds rate; annual mean of monthly observations",
    },
}

RAW_DIR, OUTPUT_PATH

## 2. Helper function for reading FRED CSV files

FRED CSV downloads usually have two columns: `observation_date` plus the series ID. This function is flexible enough to handle common date column names.

In [ ]:
def read_fred_series(series_id: str, filename: str, clean_name: str) -> pd.DataFrame:
    path = RAW_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing raw file: {path}\n"
            f"Save the FRED download as {filename} under data/raw/fred/."
        )

    df = pd.read_csv(path)
    df.columns = [str(col).strip() for col in df.columns]

    date_candidates = ["observation_date", "DATE", "date", "Date"]
    date_col = next((col for col in date_candidates if col in df.columns), None)
    if date_col is None:
        date_col = df.columns[0]

    value_candidates = [series_id, clean_name]
    value_col = next((col for col in value_candidates if col in df.columns), None)
    if value_col is None:
        non_date_cols = [col for col in df.columns if col != date_col]
        if len(non_date_cols) != 1:
            raise ValueError(f"Could not identify value column for {series_id}. Columns: {list(df.columns)}")
        value_col = non_date_cols[0]

    out = df[[date_col, value_col]].copy()
    out.columns = ["date", clean_name]
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out[clean_name] = pd.to_numeric(out[clean_name], errors="coerce")
    out = out.dropna(subset=["date"])
    out = out[(out["date"] >= START_DATE) & (out["date"] <= END_DATE)]
    out["series_id"] = series_id

    return out[["date", "series_id", clean_name]]

## 3. Load raw FRED data

In [ ]:
raw_series = {}

for series_id, meta in SERIES.items():
    raw_series[series_id] = read_fred_series(
        series_id=series_id,
        filename=meta["filename"],
        clean_name=meta["clean_name"],
    )

for series_id, df in raw_series.items():
    display(series_id, df.head(), df.tail())

## 4. Check date coverage

Use this table to identify partial-year coverage, especially for 2025.

In [ ]:
coverage_rows = []

for series_id, df in raw_series.items():
    value_col = SERIES[series_id]["clean_name"]
    coverage_rows.append(
        {
            "series_id": series_id,
            "clean_name": value_col,
            "first_date": df["date"].min(),
            "last_date": df["date"].max(),
            "n_observations": df[value_col].notna().sum(),
            "n_missing_values": df[value_col].isna().sum(),
        }
    )

coverage = pd.DataFrame(coverage_rows)
coverage

## 5. Convert each series to annual means

In [ ]:
annual_frames = []

for series_id, df in raw_series.items():
    value_col = SERIES[series_id]["clean_name"]
    annual = (
        df.assign(year=df["date"].dt.year)
        .groupby("year", as_index=False)
        .agg(
            **{
                value_col: (value_col, "mean"),
                f"{value_col}_n_obs": (value_col, "count"),
                f"{value_col}_first_date": ("date", "min"),
                f"{value_col}_last_date": ("date", "max"),
            }
        )
    )
    annual_frames.append(annual)

fred_annual = annual_frames[0]
for annual in annual_frames[1:]:
    fred_annual = fred_annual.merge(annual, on="year", how="outer")

fred_annual = fred_annual.sort_values("year").reset_index(drop=True)
fred_annual

## 6. Compute inflation from annual CPI

`inflation_pct` is computed as the percent change in annual average CPI.

In [ ]:
fred_annual["inflation_pct"] = fred_annual["cpi"].pct_change() * 100

output_cols = [
    "year",
    "mortgage_rate_pct",
    "cpi",
    "inflation_pct",
    "interest_rate_pct",
]

fred_clean = fred_annual[output_cols].copy()
fred_clean

## 7. Basic validation checks

In [ ]:
expected_years = set(range(2015, 2026))
actual_years = set(fred_clean["year"])
missing_years = sorted(expected_years - actual_years)
extra_years = sorted(actual_years - expected_years)

print(f"Missing years: {missing_years}")
print(f"Extra years: {extra_years}")

display(fred_clean.describe(include="all"))
display(fred_clean[fred_clean.isna().any(axis=1)])

## 8. Review observation counts

This table is useful for documenting whether a year is complete. For example, a full year of monthly data usually has 12 observations, while a full year of weekly mortgage-rate data usually has about 52 observations.

In [ ]:
count_cols = [col for col in fred_annual.columns if col.endswith("_n_obs")]
date_cols = [col for col in fred_annual.columns if col.endswith("_first_date") or col.endswith("_last_date")]

fred_annual[["year"] + count_cols + date_cols]

## 9. Export cleaned annual FRED table

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
fred_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Saved cleaned FRED data to: {OUTPUT_PATH}")

## 10. Suggested source note text

Copy or adapt this into `data/metadata/economic_labor_source_notes.md` when you document the FRED portion.

```text
FRED economic variables were downloaded for 2015-01-01 through 2025-12-31. The raw files are stored in data/raw/fred/ and were not modified. MORTGAGE30US is a weekly national 30-year fixed mortgage rate and was converted to an annual mean. CPIAUCSL is a monthly national CPI index and was converted to an annual mean. FEDFUNDS is a monthly national effective federal funds rate and was converted to an annual mean. inflation_pct was derived as the year-over-year percent change in annual average CPI. These variables are national controls and do not vary by Ohio market.
```
